In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from matplotlib import pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
import re


In [2]:
# Load the data
data = pd.read_excel('data-spell-checker.xlsx')
data.head()

,word,label
0,අභිචෝදකයා,1
1,අංකනය,1
2,අංකන,1
3,අංකය,1
4,අංකාන්තරය,1


In [3]:
# Data preprocessing
X = data['word'].apply(lambda x: x.lower())  # Lowercasing
X = X.apply(lambda x: re.sub(r'[^\w\s]', '', x))  # Remove special characters
y = data['label']


In [4]:
pip install memory_profiler


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 24.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from memory_profiler import profile

In [7]:
# Load the dataset
data = pd.read_excel('data-spell-checker.xlsx')

# Data preprocessing
X = data['word'].apply(lambda x: x.lower())  # Lowercasing
y = data['label']

In [8]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Feature extraction using Bag-of-Words with limited features
vectorizer = CountVectorizer(max_features=5000)  # Limiting to 5,000 features
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)



In [10]:
# Dimensionality reduction using Truncated SVD
svd = TruncatedSVD(n_components=500)  # Adjust the number of components as needed
X_train_reduced = svd.fit_transform(X_train_vectorized)
X_test_reduced = svd.transform(X_test_vectorized)



In [11]:
# Sample a subset of the data
sample_percent = 0.5  # 50% of the data
random_indices = np.random.choice(X_train_reduced.shape[0], int(X_train_reduced.shape[0] * sample_percent), replace=False)
X_train_reduced_sampled = X_train_reduced[random_indices]
y_train_sampled = y_train.iloc[random_indices]



In [12]:
# Train and evaluate Gaussian Processes with memory profiling
@profile
def train_gaussian_process(X_train, y_train):
    # Train Gaussian Process classifier
    gp_classifier = GaussianProcessClassifier(kernel=RBF())
    gp_classifier.fit(X_train, y_train)
    return gp_classifier



In [14]:
from sklearn.ensemble import RandomForestClassifier

# Train RandomForestClassifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_reduced_sampled, y_train_sampled)

# Predict using trained classifier
rf_predictions = rf_classifier.predict(X_test_reduced)
rf_accuracy = accuracy_score(y_test, rf_predictions)
print("Random Forest Accuracy:", rf_accuracy)
print(classification_report(y_test, rf_predictions))


Random Forest Accuracy: 0.619163179916318
              precision    recall  f1-score   support

           0       0.64      0.79      0.71      7025
           1       0.56      0.38      0.45      4925

    accuracy                           0.62     11950
   macro avg       0.60      0.58      0.58     11950
weighted avg       0.61      0.62      0.60     11950

